# Results Evaluation
**WESAD RF Baseline (Binary: Stress vs Non-Stress)**

Loads all saved LOSOCV results and produces publication-ready
evaluation outputs:

1. Per-fold metrics tables (accuracy, F1-weighted, F1-macro, per subject)
2. Confusion matrices (aggregate + per-subject heatmaps)
3. FS ablation comparison bar charts
4. Windowing ablation comparison
5. Normalization ablation comparison
6. LOSOCV subject variability (box plots)
7. Comparison table vs literature

## 1. Setup & Load Results

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 140
plt.rcParams["figure.facecolor"] = "white"

RESULTS_BASE = "results/binary/"
META_COLS = ["sid", "label", "window_idx", "is_overlapped"]


def load_results(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return None

In [ ]:
# Load FS ablation results
fs_methods = ["none", "tree", "kbest", "pca"]
fs_results = {}

for dev in ["chest", "wrist"]:
    fs_results[dev] = {}
    for method in fs_methods:
        path = os.path.join(
            RESULTS_BASE, f"{dev}_fs_ablation", f"fs_{method}", "losocv_results.pkl"
        )
        res = load_results(path)
        if res:
            fs_results[dev][method] = res
            agg = res["aggregate"]
            print(
                f"  {dev}/{method}: F1m={agg['f1_macro_mean']:.4f}+/-{agg['f1_macro_std']:.4f}"
            )
        else:
            print(f"  {dev}/{method}: NOT FOUND")

# Load windowing ablation results
SLIDE_VALUES = [0.25, 0.5, 5.0, 20.0, 60.0]
window_results = {}

for dev in ["chest", "wrist"]:
    window_results[dev] = {}
    for slide in SLIDE_VALUES:
        path = os.path.join(
            RESULTS_BASE,
            "window_ablation",
            dev,
            f"slide_{slide}s",
            "losocv_results.pkl",
        )
        res = load_results(path)
        if res:
            window_results[dev][slide] = res

    loaded = list(window_results[dev].keys())
    print(f"  {dev} window ablation: loaded slides {loaded}")

# Load normalization ablation results
norm_results = {}
for dev in ["chest", "wrist"]:
    path = os.path.join(
        RESULTS_BASE, "norm_ablation", dev, "unnormalized", "losocv_results.pkl"
    )
    res = load_results(path)
    if res:
        norm_results[dev] = res
        agg = res["aggregate"]
        print(
            f"  {dev} unnormalized: F1m={agg['f1_macro_mean']:.4f}+/-{agg['f1_macro_std']:.4f}"
        )
    else:
        print(f"  {dev} unnormalized: NOT FOUND")

print("\nAll available results loaded.")

## 2. Per-Fold Metrics Table

Detailed per-subject performance for the primary configuration
(tree FS, 5s slide, normalized). This shows which subjects are
easy or hard and exposes LOSOCV variability.

In [ ]:
# Per-fold table for primary config (tree FS)
for dev in ["chest", "wrist"]:
    if "tree" not in fs_results[dev]:
        print(f"{dev}: tree FS results not found")
        continue

    res = fs_results[dev]["tree"]
    folds = res["fold_results"]

    rows = []
    for f in folds:
        rows.append(
            {
                "Subject": f"S{f['subject']}",
                "Accuracy": f["accuracy"],
                "F1 Weighted": f["f1_weighted"],
                "F1 Macro": f["f1_macro"],
                "Precision": f["precision"],
                "Recall": f["recall"],
                "N Features": f["n_features"],
                "N Val": f["n_val"],
                "Fold Time (s)": f["fold_time"],
            }
        )

    df_folds = pd.DataFrame(rows)

    # Add aggregate rows
    agg = res["aggregate"]
    df_folds.loc[len(df_folds)] = {
        "Subject": "MEAN",
        "Accuracy": agg["accuracy_mean"],
        "F1 Weighted": agg["f1_weighted_mean"],
        "F1 Macro": agg["f1_macro_mean"],
        "Precision": agg["precision_mean"],
        "Recall": agg["recall_mean"],
        "N Features": "",
        "N Val": "",
        "Fold Time (s)": "",
    }
    df_folds.loc[len(df_folds)] = {
        "Subject": "STD",
        "Accuracy": agg["accuracy_std"],
        "F1 Weighted": agg["f1_weighted_std"],
        "F1 Macro": agg["f1_macro_std"],
        "Precision": agg["precision_std"],
        "Recall": agg["recall_std"],
        "N Features": "",
        "N Val": "",
        "Fold Time (s)": "",
    }

    print(f"\n{dev.upper()} — Per-Fold LOSOCV Results (Tree FS, Binary)")
    print("=" * 100)
    print(df_folds.to_string(index=False, float_format="%.4f"))

## 3. Confusion Matrices

In [ ]:
# Aggregate confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = ["Non-Stress", "Stress"]

for ax, dev in zip(axes, ["chest", "wrist"]):
    if "tree" not in fs_results[dev]:
        ax.text(
            0.5,
            0.5,
            f"No results for {dev}",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        continue

    res = fs_results[dev]["tree"]
    cm = res["global_confusion_matrix"]

    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(
        f"{dev.capitalize()} — Aggregate Confusion Matrix (Tree FS)", fontsize=12
    )

plt.suptitle("Binary LOSOCV — Global Confusion Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print classification reports
for dev in ["chest", "wrist"]:
    if "tree" in fs_results[dev]:
        print(f"\n{dev.upper()} — Global Classification Report:")
        print(fs_results[dev]["tree"]["global_classification_report"])

In [ ]:
# Per-subject confusion matrix heatmaps
for dev in ["chest", "wrist"]:
    if "tree" not in fs_results[dev]:
        continue

    folds = fs_results[dev]["tree"]["fold_results"]
    n_folds = len(folds)
    ncols = 5
    nrows = (n_folds + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3 * nrows))
    axes = axes.flatten()

    for i, f in enumerate(folds):
        cm = f["confusion_matrix"]
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            xticklabels=class_names,
            yticklabels=class_names,
            ax=axes[i],
            annot_kws={"size": 10},
        )
        acc = f["accuracy"]
        f1m = f["f1_macro"]
        axes[i].set_title(f"S{f['subject']} (Acc={acc:.2f}, F1m={f1m:.2f})", fontsize=9)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{dev.capitalize()} — Per-Subject Confusion Matrices (Tree FS)",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

## 4. Feature Selection Ablation Comparison

In [ ]:
# FS ablation bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
method_colors = {
    "none": "#9E9E9E",
    "tree": "#4CAF50",
    "kbest": "#2196F3",
    "pca": "#FF9800",
}

for ax, dev in zip(axes, ["chest", "wrist"]):
    methods_available = [m for m in fs_methods if m in fs_results[dev]]
    if not methods_available:
        continue

    f1m_means = [
        fs_results[dev][m]["aggregate"]["f1_macro_mean"] for m in methods_available
    ]
    f1m_stds = [
        fs_results[dev][m]["aggregate"]["f1_macro_std"] for m in methods_available
    ]
    acc_means = [
        fs_results[dev][m]["aggregate"]["accuracy_mean"] for m in methods_available
    ]

    x = np.arange(len(methods_available))
    bars = ax.bar(
        x,
        f1m_means,
        yerr=f1m_stds,
        capsize=5,
        color=[method_colors[m] for m in methods_available],
        edgecolor="white",
        width=0.6,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(methods_available)
    ax.set_ylabel("F1 Macro (mean +/- std)")
    ax.set_title(f"{dev.capitalize()} — FS Method Comparison")
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3, axis="y")

    for bar, mean, std, acc in zip(bars, f1m_means, f1m_stds, acc_means):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std + 0.02,
            f"F1m={mean:.3f}\nAcc={acc:.3f}",
            ha="center",
            fontsize=8,
            fontweight="bold",
        )

plt.suptitle(
    "Feature Selection Ablation — F1 Macro (Binary, LOSOCV)", fontsize=14, y=1.02
)
plt.tight_layout()
plt.show()

# Table
rows = []
for dev in ["chest", "wrist"]:
    for m in fs_methods:
        if m in fs_results[dev]:
            agg = fs_results[dev][m]["aggregate"]
            rows.append(
                {
                    "Device": dev.capitalize(),
                    "FS Method": m,
                    "Accuracy": f"{agg['accuracy_mean']:.4f} +/- {agg['accuracy_std']:.4f}",
                    "F1 Weighted": f"{agg['f1_weighted_mean']:.4f} +/- {agg['f1_weighted_std']:.4f}",
                    "F1 Macro": f"{agg['f1_macro_mean']:.4f} +/- {agg['f1_macro_std']:.4f}",
                }
            )
print("\nFS ABLATION SUMMARY")
print(pd.DataFrame(rows).to_string(index=False))

## 5. Windowing Ablation Comparison

In [ ]:
# Windowing ablation line charts
if any(window_results[d] for d in window_results):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    dev_colors = {"chest": "#2196F3", "wrist": "#FF9800"}
    metrics = [
        ("accuracy_mean", "accuracy_std", "Accuracy"),
        ("f1_weighted_mean", "f1_weighted_std", "F1 Weighted"),
        ("f1_macro_mean", "f1_macro_std", "F1 Macro"),
    ]

    for ax, (mean_key, std_key, label) in zip(axes, metrics):
        for dev in ["chest", "wrist"]:
            if not window_results[dev]:
                continue
            slides = sorted(window_results[dev].keys())
            means = [window_results[dev][s]["aggregate"][mean_key] for s in slides]
            stds = [window_results[dev][s]["aggregate"][std_key] for s in slides]
            ax.errorbar(
                slides,
                means,
                yerr=stds,
                fmt="o-",
                color=dev_colors[dev],
                linewidth=2,
                markersize=8,
                capsize=5,
                label=dev.capitalize(),
            )
        ax.set_xlabel("Slide size (s)")
        ax.set_ylabel(label)
        ax.set_title(f"{label} vs Slide Size")
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1.05)

    plt.suptitle("Windowing Ablation — Binary LOSOCV", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Table
    rows = []
    for dev in ["chest", "wrist"]:
        for slide in sorted(window_results[dev].keys()):
            agg = window_results[dev][slide]["aggregate"]
            rows.append(
                {
                    "Device": dev.capitalize(),
                    "Slide (s)": slide,
                    "Accuracy": f"{agg['accuracy_mean']:.4f} +/- {agg['accuracy_std']:.4f}",
                    "F1 Macro": f"{agg['f1_macro_mean']:.4f} +/- {agg['f1_macro_std']:.4f}",
                }
            )
    print("\nWINDOWING ABLATION SUMMARY")
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print("No windowing ablation results found.")

## 6. LOSOCV Subject Variability

Box plots showing the distribution of per-fold metrics across subjects.
Subjects 2 and 11 are known outliers in the WESAD literature.

In [ ]:
# Subject variability box plots
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, dev in zip(axes, ["chest", "wrist"]):
    if "tree" not in fs_results[dev]:
        continue

    folds = fs_results[dev]["tree"]["fold_results"]
    subjects = [f"S{f['subject']}" for f in folds]
    acc = [f["accuracy"] for f in folds]
    f1w = [f["f1_weighted"] for f in folds]
    f1m = [f["f1_macro"] for f in folds]

    x = np.arange(len(subjects))
    width = 0.25

    ax.bar(x - width, acc, width, color="#2196F3", alpha=0.8, label="Accuracy")
    ax.bar(x, f1w, width, color="#FF9800", alpha=0.8, label="F1 Weighted")
    ax.bar(x + width, f1m, width, color="#4CAF50", alpha=0.8, label="F1 Macro")

    ax.set_xticks(x)
    ax.set_xticklabels(subjects, fontsize=8, rotation=45)
    ax.set_ylabel("Score")
    ax.set_title(f"{dev.capitalize()} — Per-Subject Performance (Tree FS)")
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    # Mark outliers S2, S11
    for i, f in enumerate(folds):
        if f["subject"] in [2, 11]:
            ax.annotate(
                "*",
                (i, max(acc[i], f1w[i], f1m[i]) + 0.02),
                ha="center",
                fontsize=14,
                color="red",
                fontweight="bold",
            )

    # Add mean line
    ax.axhline(y=np.mean(f1m), color="#4CAF50", linestyle="--", alpha=0.5, linewidth=1)

plt.suptitle(
    "Per-Subject LOSOCV (known outliers S2, S11 marked *)", fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# Box plot of metric distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dev in zip(axes, ["chest", "wrist"]):
    if "tree" not in fs_results[dev]:
        continue

    folds = fs_results[dev]["tree"]["fold_results"]
    data = {
        "Accuracy": [f["accuracy"] for f in folds],
        "F1 Weighted": [f["f1_weighted"] for f in folds],
        "F1 Macro": [f["f1_macro"] for f in folds],
        "Precision": [f["precision"] for f in folds],
        "Recall": [f["recall"] for f in folds],
    }
    df_box = pd.DataFrame(data)

    bp = df_box.boxplot(
        ax=ax,
        patch_artist=True,
        return_type="dict",
        boxprops={"facecolor": "#2196F3", "alpha": 0.6},
        medianprops={"color": "black", "linewidth": 2},
    )
    ax.set_ylabel("Score")
    ax.set_title(f"{dev.capitalize()} — Metric Distribution Across 15 Folds")
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()